# Lesson 1.3.3 - Basic REST APIs with HTTP & JSON

## Objectives
- Call public API endpoints.
- Parse/filter JSON responses.
- Handle errors, timeouts, and retries.


In [ ]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


In [ ]:
BASE_URL = "https://jsonplaceholder.typicode.com"
r = requests.get(f"{BASE_URL}/posts/1", timeout=10)
print(r.status_code)
post = r.json()
print(post["title"])
print(post["body"][:80])


In [ ]:
posts = requests.get(f"{BASE_URL}/posts", timeout=10).json()
print("Total posts:", len(posts))
user_1_posts = [p for p in posts if p["userId"] == 1]
print("User 1 posts:", len(user_1_posts))


In [ ]:
def build_session(total_retries=3):
    retry = Retry(total=total_retries, backoff_factor=0.4, status_forcelist=[429,500,502,503,504], allowed_methods=["GET", "POST", "PUT", "DELETE", "PATCH"])
    adapter = HTTPAdapter(max_retries=retry)
    s = requests.Session()
    s.mount("http://", adapter)
    s.mount("https://", adapter)
    return s

def fetch_json(url, session, timeout=10):
    try:
        resp = session.get(url, timeout=timeout)
        resp.raise_for_status()
    except requests.exceptions.RequestException as exc:
        raise RuntimeError(f"Request failed for {url}: {exc}") from exc
    try:
        return resp.json()
    except ValueError as exc:
        raise RuntimeError(f"Invalid JSON from {url}") from exc

session = build_session()
comments = fetch_json(f"{BASE_URL}/comments?postId=1", session)
print(len(comments), comments[0]["email"])


In [ ]:
try:
    bad_resp = requests.get(f"{BASE_URL}/nonexistent", timeout=5)
    print("status:", bad_resp.status_code)
except requests.exceptions.RequestException as exc:
    print(exc)

try:
    fetch_json("https://this-domain-should-not-exist.invalid", session, timeout=3)
except RuntimeError as exc:
    print(exc)


## AI Engineering Context
Same patterns apply to LLM/model APIs: timeout, retry, schema checks, and structured error logging.


## Business Case Studies & Exceptions
- Timeout cascades in feature APIs can block inference workers.
- Schema drift in external API can break parser and downstream model.
- Always validate status codes and response shape before inference.


## Interview Questions & Answers
1. **Q:** How call API safely in Python?  
   **A:** timeout + `raise_for_status` + exception handling + JSON validation.
2. **Q:** 200 vs 404 vs 500?  
   **A:** success, missing resource, server error.
3. **Q:** Why retries with cap?  
   **A:** Avoid retry storms.
4. **Q:** Why check status before `.json()`?  
   **A:** Error payload may not match expected schema.
5. **Q:** What should be logged on API failure?  
   **A:** endpoint, status, latency, request ID, exception type.
6. **Q:** Why use timeout always?  
   **A:** Prevent thread/resource starvation.
7. **Q:** What is idempotency?  
   **A:** Repeat request has same effect.
8. **Q:** How protect API keys?  
   **A:** Store in environment/secret manager.
9. **Q:** Why schema validation in AI APIs?  
   **A:** Prevent bad payloads from poisoning predictions.
10. **Q:** REST role in production AI?  
    **A:** Service integration for features, inference, and monitoring.
